# Stage 1: CKA drift and gradient cosine per level (from checkpoints)

No training; CPU runtime is fine. Replaces the team's in-loop structural plugin with post-hoc measures on the saved
per-task checkpoints, using fixed image sets (definitions in `src/stage1_structural.py`):

- **`cka_to_final`** per level: how similar the final model's features of task *j*'s test images are to the features
  right after learning *j* (1 = unchanged; lower = more drift). The representation-similarity partner of probe forgetting.
- **`cka_consecutive`** per level: similarity between consecutive checkpoints on a fixed 1000-image test subset.
- **`grad_cos`** per parameter group (3 depth groups + head): cosine between the gradients of the previous task and the
  new task at the same checkpoint. Negative = the new task pulls the group against the old one.

Levels are layer1 / layer3 / layer4 (pooled), as in the probe analysis. Cost per training run: not measured yet.

In [ ]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # <-- EDIT ONCE
BRANCH = "nested-b-d"   # the branch with the code this notebook needs (set back to "main" after merging)

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", url, CODE], check=True)
# fetch + checkout + pull, so a clone left on another branch earlier in the session switches over
subprocess.run(["git", "-C", CODE, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", CODE, "pull", "-q", "origin", BRANCH], check=True)
print(BRANCH, subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
_MINE = {"fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic",
         "stage1_structural", "nested_builder"}
for _m in list(sys.modules):
    if _m in _MINE or _m == "src" or _m.startswith("src."):   # src.* = the team's nested code
        sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = False   # analysis only: CPU runtime is fine
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

In [ ]:
import stage1_analysis as sa
import stage1_structural as st
STAGES = ["stage1_control", "stage1_nested_control"]
TRAIN_RUNS = sa.list_runs(paths, stage=STAGES)
for r in TRAIN_RUNS:
    c = r["config"]; print(r["run_id"], sr.arm_label(c), "lr", c["lr"], "seed", c["seed"])
SCFGS = st.structural_configs(TRAIN_RUNS)

In [ ]:
out = run_sweep(SCFGS, st.structural_run, paths)

In [ ]:
import numpy as np, pandas as pd
rows = []
for a in SCFGS:
    r = load_result(a, paths)
    if r is None: continue
    m = r["metrics"]; c = m["train_config"]
    row = {"arm": sr.arm_label(c), "lr": c["lr"], "seed": c["seed"]}
    for l, name in enumerate(m["level_modules"]):
        row[f"cka_to_final/{name}"] = m["cka_to_final_avg"][l]
        row[f"cka_consecutive/{name}"] = m["cka_consecutive_avg"][l]
    # group names differ between the two model classes (conv1+bn1+layer1 vs stem+layer1), so label by position
    n_g = len(m["grad_cos_avg"])
    for g, v in enumerate(m["grad_cos_avg"]):
        row["grad_cos/" + ("head" if g == n_g - 1 else f"group{g}")] = v
    rows.append(row)
sdf = pd.DataFrame(rows)
display(sdf.drop(columns="seed").groupby(["arm", "lr"]).agg(["mean", "std"]).T.round(3))

**How to read it.** The thesis predicts that slow levels keep a higher `cka_to_final` than the same levels in Flat.
Differences smaller than the seed std are not meaningful with 3 seeds. `grad_cos` near 0 or negative at a level means
consecutive tasks pull that level in unrelated or opposing directions.